In [1]:
import pyspark
from pyspark.sql import SparkSession

print(f"Versión de PySpark: {pyspark.__version__}")

Versión de PySpark: 3.5.1


In [2]:
spark = SparkSession.builder \
    .appName("FinancialPipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(spark)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 22:15:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Leer las transacciones
df_transactions = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("../data/raw/train_transaction.csv")

print(f"Filas: {df_transactions.count()}")
print(f"Columnas: {len(df_transactions.columns)}")

Filas: 590540
Columnas: 394


In [4]:
# Ver los tipos de datos de las primeras columnas
df_transactions.printSchema()

root
 |-- TransactionID: integer (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- TransactionDT: integer (nullable = true)
 |-- TransactionAmt: double (nullable = true)
 |-- ProductCD: string (nullable = true)
 |-- card1: integer (nullable = true)
 |-- card2: double (nullable = true)
 |-- card3: double (nullable = true)
 |-- card4: string (nullable = true)
 |-- card5: double (nullable = true)
 |-- card6: string (nullable = true)
 |-- addr1: double (nullable = true)
 |-- addr2: double (nullable = true)
 |-- dist1: double (nullable = true)
 |-- dist2: double (nullable = true)
 |-- P_emaildomain: string (nullable = true)
 |-- R_emaildomain: string (nullable = true)
 |-- C1: double (nullable = true)
 |-- C2: double (nullable = true)
 |-- C3: double (nullable = true)
 |-- C4: double (nullable = true)
 |-- C5: double (nullable = true)
 |-- C6: double (nullable = true)
 |-- C7: double (nullable = true)
 |-- C8: double (nullable = true)
 |-- C9: double (nullable = true)
 |-- C10:

In [5]:
# Ver las primeras 5 filas solo con columnas clave
columnas_clave = ["TransactionID", "TransactionDT", "TransactionAmt", 
                  "ProductCD", "card4", "card6", "isFraud"]

df_transactions.select(columnas_clave).show(5)

+-------------+-------------+--------------+---------+----------+------+-------+
|TransactionID|TransactionDT|TransactionAmt|ProductCD|     card4| card6|isFraud|
+-------------+-------------+--------------+---------+----------+------+-------+
|      2987000|        86400|          68.5|        W|  discover|credit|      0|
|      2987001|        86401|          29.0|        W|mastercard|credit|      0|
|      2987002|        86469|          59.0|        W|      visa| debit|      0|
|      2987003|        86499|          50.0|        W|mastercard| debit|      0|
|      2987004|        86506|          50.0|        H|mastercard|credit|      0|
+-------------+-------------+--------------+---------+----------+------+-------+
only showing top 5 rows



In [6]:
from pyspark.sql import functions as F

# Calcular % de nulos por columna solo en las columnas clave
columnas_clave = ["TransactionID", "TransactionDT", "TransactionAmt", 
                  "ProductCD", "card4", "card6", "isFraud",
                  "P_emaildomain", "R_emaildomain", "dist1", "dist2"]

total = df_transactions.count()

nulos = df_transactions.select([
    F.round((F.count(F.when(F.col(c).isNull(), c)) / total) * 100, 2)
     .alias(f"{c}_nulos_%")
    for c in columnas_clave
])

nulos.show()

+---------------------+---------------------+----------------------+-----------------+-------------+-------------+---------------+---------------------+---------------------+-------------+-------------+
|TransactionID_nulos_%|TransactionDT_nulos_%|TransactionAmt_nulos_%|ProductCD_nulos_%|card4_nulos_%|card6_nulos_%|isFraud_nulos_%|P_emaildomain_nulos_%|R_emaildomain_nulos_%|dist1_nulos_%|dist2_nulos_%|
+---------------------+---------------------+----------------------+-----------------+-------------+-------------+---------------+---------------------+---------------------+-------------+-------------+
|                  0.0|                  0.0|                   0.0|              0.0|         0.27|         0.27|            0.0|                15.99|                76.75|        59.65|        93.63|
+---------------------+---------------------+----------------------+-----------------+-------------+-------------+---------------+---------------------+---------------------+-------------+

In [7]:
df_transactions.select("TransactionAmt", "isFraud") \
    .describe() \
    .show()

+-------+------------------+-------------------+
|summary|    TransactionAmt|            isFraud|
+-------+------------------+-------------------+
|  count|            590540|             590540|
|   mean|135.02717637250706|0.03499000914417313|
| stddev| 239.1625220137339|0.18375463417841392|
|    min|             0.251|                  0|
|    max|         31937.391|                  1|
+-------+------------------+-------------------+



In [8]:
df_transactions.groupBy("isFraud") \
    .count() \
    .withColumn("porcentaje", F.round((F.col("count") / total) * 100, 2)) \
    .orderBy("isFraud") \
    .show()


+-------+------+----------+
|isFraud| count|porcentaje|
+-------+------+----------+
|      0|569877|      96.5|
|      1| 20663|       3.5|
+-------+------+----------+

